# 🌦️ 05 — Preprocessing & visualisation : Météo × Y

Sous-ensemble dédié de `df_model.parquet` : les **21 variables météo**
construites dans `01a_pipeline_meteo.ipynb` + les **9 variables cibles (Y)**
(3 indicateurs `urgences`/`hosp`/`sos` × 3 pathologies `allergie`/`asthme`/`bronchiolite`).

Objectif : avant de passer à la modélisation, isoler ce sous-ensemble pour
1. vérifier les valeurs manquantes (NA),
2. repérer les valeurs extrêmes / suspectes et décider quoi en faire,
3. visualiser distributions, saisonnalité et corrélations météo × Y.

Prérequis : `02_merge_final.ipynb` (produit `data/processed/df_model.parquet`).

In [58]:
import os
import sys
from pathlib import Path
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore")

from src.config import TABLES_DIR

df = pd.read_parquet(TABLES_DIR / "df_model.parquet")
df["annee_mois_dt"] = pd.to_datetime(df["annee_mois"] + "-01")

print(f"df_model chargé : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
print(f"Période : {df['annee_mois'].min()} → {df['annee_mois'].max()}")

df_model chargé : 6,912 lignes × 83 colonnes
Période : 2020-01 → 2025-12


---
## 1. Sélection des variables

- **Météo** : les 21 colonnes produites par `build_dim_meteo()` (cf. `01a_pipeline_meteo.ipynb`
  pour la justification de chaque variable).
- **Y** : les 9 colonnes `taux_<indicateur>_<pathologie>` (part des passages aux
  urgences / hospitalisations post-urgences / actes SOS Médecins, pour
  allergie / asthme / bronchiolite — **pas des taux d'incidence en population**,
  cf. dictionnaire de données).
- **Clés** : `dept`, `annee_mois` (+ dérivés `annee`, `mois` déjà présents dans `df_model`).

In [59]:
KEYS = ["dept", "annee_mois", "annee_mois_dt", "annee", "mois"]

METEO_COLS = [
    "temp_moy", "temp_max", "temp_min", "amplitude_thermique_moy",
    "nb_jours_gelee", "nb_jours_sans_degel", "nb_jours_chaud25", "nb_jours_chaud30",
    "humidite_moy", "vent_moy", "vent_rafale_max", "nb_jours_vent_fort",
    "precip_total", "nb_jours_pluie1mm", "nb_jours_pluie5mm",
    "ensoleillement_total", "rayonnement_total",
    "nb_jours_orage", "nb_jours_brouillard", "pression_moy", "etp_total",
]

PATHOLOGIES = ["allergie", "asthme"] # on supprime bronchiolite : age constant
INDICATEURS = {"urgences": "Urgences", "hosp": "Hospitalisations"} # on supprime "SOS médecins" : taux de remplissage très faible -50%, pas exploitable
Y_COLS = [f"taux_{ind}_{p}" for ind in INDICATEURS for p in PATHOLOGIES]

# Convention couleur reprise de 03_eda.ipynb / app.py (cohérence visuelle du projet)
COLORS = {"allergie": "#E84837", "asthme": "#2980B9"}
MOIS_LABELS = ["Jan", "Fév", "Mar", "Avr", "Mai", "Jun",
               "Jul", "Aoû", "Sep", "Oct", "Nov", "Déc"]

df_sub = df[KEYS + METEO_COLS + Y_COLS].copy()
print(f"df_sub : {df_sub.shape[0]:,} lignes × {df_sub.shape[1]} colonnes "
      f"({len(METEO_COLS)} météo + {len(Y_COLS)} Y + {len(KEYS)} clés)")

df_sub : 6,912 lignes × 30 colonnes (21 météo + 4 Y + 5 clés)


---
## 2. Valeurs manquantes (NA)

In [60]:
na_count = df_sub[METEO_COLS + Y_COLS].isna().sum()
na_pct = (na_count / len(df_sub) * 100).round(1)
na_table = pd.DataFrame({"n_manquant": na_count, "pct_manquant": na_pct}).sort_values(by="pct_manquant", ascending=False)

display(na_table)

,n_manquant,pct_manquant
nb_jours_orage,1203,17.4
pression_moy,432,6.2
nb_jours_brouillard,355,5.1
rayonnement_total,317,4.6
ensoleillement_total,313,4.5
humidite_moy,216,3.1
vent_moy,216,3.1
vent_rafale_max,216,3.1
nb_jours_vent_fort,216,3.1
taux_hosp_asthme,5,0.1


**Commentaire** :
- Les NA météo (3 à 17 %) reflètent une **couverture de stations incomplète**
  (toutes les stations ne mesurent pas vent/humidité/rayonnement/pression —
  cf. le contrôle qualité dans `01a_pipeline_meteo.ipynb`), pas un problème de
  qualité de données. `nb_jours_orage` est la plus touchée (~17 %).

---
## 3. Valeurs extrêmes / outliers

In [61]:
desc = df_sub[METEO_COLS].describe().T[["min", "25%", "50%", "75%", "max", "mean"]]
display(desc.round(2))

# Bornes physiquement impossibles (défensif — sanity check, pas de suppression a priori)
bornes_min0 = ["nb_jours_gelee", "nb_jours_sans_degel", "nb_jours_chaud25",           "nb_jours_chaud30",
               "vent_moy", "vent_rafale_max", "nb_jours_vent_fort", "precip_total",
               "nb_jours_pluie1mm", "nb_jours_pluie5mm", "ensoleillement_total",
               "rayonnement_total", "nb_jours_orage", "nb_jours_brouillard", "etp_total"]
anomalies = []
for c in bornes_min0:
    n_neg = (df_sub[c] < 0).sum()
    if n_neg:
        anomalies.append((c, "valeur négative", n_neg))
n_hum = (df_sub["humidite_moy"] > 100).sum()
if n_hum:
    anomalies.append(("humidite_moy", "> 100%", n_hum))

print("Anomalies physiquement impossibles :", anomalies or [0])

,min,25%,50%,75%,max,mean
temp_moy,-3.68,7.62,12.10,17.88,26.52,12.61
temp_max,-0.31,11.56,17.00,23.40,34.18,17.47
temp_min,-7.21,3.61,7.16,12.28,20.27,7.74
amplitude_thermique_moy,3.70,7.71,9.88,11.70,18.00,9.73
nb_jours_gelee,0.00,0.00,0.68,6.12,27.71,3.68
nb_jours_sans_degel,0.00,0.00,0.00,0.00,14.59,0.30
nb_jours_chaud25,0.00,0.00,0.68,10.52,31.00,6.02
nb_jours_chaud30,0.00,0.00,0.00,2.41,28.58,2.14
humidite_moy,46.33,70.22,76.50,83.60,95.27,76.43
vent_moy,1.42,2.62,3.09,3.68,9.01,3.22


Anomalies physiquement impossibles : [0]


In [62]:
ncols, nrows = 4, -(-len(METEO_COLS) // 4)
fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=METEO_COLS,
                     horizontal_spacing=0.05, vertical_spacing=0.07)
for i, col in enumerate(METEO_COLS):
    r, c = divmod(i, ncols)
    # https://plotly.com/python/reference/box/
    # https://plotly.com/python/box-plots/
    fig.add_trace(go.Box(y=df_sub[col], name=col, showlegend=False,
                          marker_color="#2980B9",
                          boxpoints="outliers",   # affiche les points hors moustaches (pas "all" -> trop de points superposés)
                          boxmean=True),          # trait pointillé = moyenne, à comparer à la médiane (ligne pleine)
                  row=r + 1, col=c + 1)
fig.update_xaxes(showticklabels=False)
fig.update_annotations(font_size=11) 
fig.update_layout(height=260 * nrows, width=1400, template="plotly_white",
                   font=dict(size=11),
                   margin=dict(t=90),
                   title=dict(
                       text="Distribution & valeurs extrêmes : variables météo ",
                       # points = valeurs hors moustaches, tirets = moyenne
                       font=dict(size=15),
                   ))
fig.show()

### Vérifier ces valeurs extrêmes sont réelles (pas des erreurs)

Trois vérifications, dans l'ordre :

1. **Bornes logiques/physiques calculables** — un nombre de jours ne peut pas
   dépasser le nombre de jours du mois, `nb_jours_chaud30` ne peut pas dépasser
   `nb_jours_chaud25`, l'ensoleillement ne peut pas
   dépasser le record français (~470h/mois).
2. **Localiser le `(dept, annee_mois)` de chaque valeur extrême** et vérifier
   que la saison/la géographie sont cohérentes (ex. un pic de `nb_jours_gelee`
   doit tomber en hiver dans un département montagneux, pas en été en Corse).
3. **Recouper avec des événements météo réels connus** — le test le plus
   convaincant : est-ce que la ligne correspond à un épisode déjà documenté
   (canicule, vague de froid, épisode cévenol...) ?

In [57]:
# ── Vérification 1 : bornes logiques/physiques calculables ─────────────────
jours_du_mois = pd.to_datetime(df_sub["annee_mois"] + "-01").dt.days_in_month

violations = []
for col in ["nb_jours_gelee", "nb_jours_sans_degel", "nb_jours_chaud25", "nb_jours_chaud30",
            "nb_jours_vent_fort", "nb_jours_pluie1mm", "nb_jours_pluie5mm",
            "nb_jours_orage", "nb_jours_brouillard"]:
    n = (df_sub[col] > jours_du_mois).sum()
    if n:
        violations.append((col, n))
print("Nb jours > jours du mois (impossible) :", violations or 0
      )

n_incoherent = (df_sub["nb_jours_chaud30"] > df_sub["nb_jours_chaud25"]).sum()
display(n_incoherent) # ok

RECORD_ENSOLEILLEMENT_FR = 470  # h/mois, record français (Marseille/Toulon, juillet)
display(df_sub[df_sub["ensoleillement_total"] == 420.87]) # Marseille : 13
n_ensoleil = (df_sub["ensoleillement_total"] > RECORD_ENSOLEILLEMENT_FR).sum()
display(n_ensoleil) # ok

Nb jours > jours du mois (impossible) : 0
0


,dept,annee_mois,annee_mois_dt,annee,mois,temp_moy,temp_max,temp_min,amplitude_thermique_moy,nb_jours_gelee,...,ensoleillement_total,rayonnement_total,nb_jours_orage,nb_jours_brouillard,pression_moy,etp_total,taux_urgences_allergie,taux_urgences_asthme,taux_hosp_allergie,taux_hosp_asthme
894,13,2022-07,2022-07-01,2022,7,26.52,33.92,19.12,14.81,0.0,...,420.87,87423.0,1.33,0.0,1016.23,223.97,803.82,621.45,218.78,748.53


0


In [ ]:
# ── Vérification 2 : localiser (dept, annee_mois) des valeurs les plus extrêmes ─
vars_extremes = ["amplitude_thermique_moy", "nb_jours_gelee", "nb_jours_sans_degel",
                  "nb_jours_chaud25", # surtout 13 marseille
                  "nb_jours_chaud30", "ensoleillement_total",
                  "nb_jours_pluie5mm", "precip_total"]

for v in vars_extremes:
    top = df_sub.nlargest(3, v)[["dept", "annee_mois", v]]
    print(f"--- {v} (top 3) ---")
    print(top.to_string(index=False))
    print()

--- amplitude_thermique_moy (top 3) ---
dept annee_mois  amplitude_thermique_moy
  04    2022-07                    18.00
  04    2024-08                    17.31
  04    2020-07                    17.14

--- nb_jours_gelee (top 3) ---
dept annee_mois  nb_jours_gelee
  05    2021-01           27.71
  05    2020-01           27.07
  73    2022-01           26.96

--- nb_jours_sans_degel (top 3) ---
dept annee_mois  nb_jours_sans_degel
  73    2021-01                14.59
  74    2021-01                11.90
  05    2021-01                11.44

--- nb_jours_chaud25 (top 3) ---
dept annee_mois  nb_jours_chaud25
  13    2022-07              31.0
  13    2024-08              31.0
  82    2022-08              31.0

--- nb_jours_chaud30 (top 3) ---
dept annee_mois  nb_jours_chaud30
  13    2022-07             28.58
  13    2024-08             28.00
  30    2022-07             27.69

--- ensoleillement_total (top 3) ---
dept annee_mois  ensoleillement_total
  13    2022-07                420.

**Décision** : on **ne supprime pas** ces valeurs "extrêmes" météo —
`nb_jours_sans_degel`, `nb_jours_chaud30`, `nb_jours_orage`, `nb_jours_gelee`
sont des variables de comptage très zero-inflated (la majorité des mois vaut 0),
donc la méthode IQR classe mécaniquement toute valeur non-nulle comme "hors IQR"
: ce n'est pas du bruit, c'est le **signal utile** (jour de canicule/orage/gel
= exactement ce qu'on veut capter pour expliquer les urgences). Seules des
valeurs physiquement impossibles (négatives, humidité>100%) auraient justifié
une correction — il n'y en a aucune ici.

---
## 4. Traitement des NA (imputation)

Stratégie : **médiane mensuelle
nationale** pour les colonnes météo (la NA vient d'un manque de couverture de
stations un mois donné, pas d'un phénomène aléatoire.
La médiane du même mois sur les autres départements est une valeur raisonnable. Les colonnes **Y ne sont pas imputées** : une cible manquante
doit rester manquante (exclue de l'entraînement), l'imputer biaiserait la
modélisation.

In [64]:
df_clean = df_sub.copy()
meteo_na_cols = [c for c in METEO_COLS if df_clean[c].isna().any()]

for col in meteo_na_cols:
    mediane = df_clean.groupby("mois")[col].transform("median")
    df_clean[col] = df_clean[col].fillna(mediane)

missing_after = df_clean[METEO_COLS].isna().sum()
missing_after = missing_after[missing_after > 0]
print("Colonnes météo imputées :", meteo_na_cols)
print("NA météo restants après imputation :", missing_after)
print("NA Y (volontairement non imputés) :")
print(df_clean[Y_COLS].isna().sum()[df_clean[Y_COLS].isna().sum() > 0])

Colonnes météo imputées : ['humidite_moy', 'vent_moy', 'vent_rafale_max', 'nb_jours_vent_fort', 'ensoleillement_total', 'rayonnement_total', 'nb_jours_orage', 'nb_jours_brouillard', 'pression_moy']
NA météo restants après imputation : Series([], dtype: int64)
NA Y (volontairement non imputés) :
taux_hosp_allergie    5
taux_hosp_asthme      5
dtype: int64


In [27]:
print(meteo_na_cols)

['humidite_moy', 'vent_moy', 'vent_rafale_max', 'nb_jours_vent_fort', 'ensoleillement_total', 'rayonnement_total', 'nb_jours_orage', 'nb_jours_brouillard', 'pression_moy']


---
## 5. Visualisation

### Distributions

In [72]:
# Variables météo qui ne peuvent pas être négatives (tout METEO_COLS sauf les
# 3 températures brutes, légitimement < 0°C en hiver) -> bins forcés à partir
# de 0 pour éviter qu'un bin déborde visuellement sous 0 (cf. discussion NA/valeurs négatives)
NON_NEGATIF = [c for c in METEO_COLS if c not in ("temp_moy", "temp_max", "temp_min")]

fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=METEO_COLS)
for i, col in enumerate(METEO_COLS):
    r, c = divmod(i, ncols)
    xbins = dict(start=0) if col in NON_NEGATIF else None
    fig.add_trace(go.Histogram(x=df_clean[col].dropna(), showlegend=False,
                                marker_color="#2980B9", xbins=xbins),
                  row=r + 1, col=c + 1)
fig.update_xaxes(showticklabels=False)
fig.update_annotations(font_size=11)
fig.update_layout(height=260 * nrows, width=1400, template="plotly_white",
                   font=dict(size=11),
                   margin=dict(t=90),
                   title=dict(text="Distributions : météo (après imputation)",
                               font=dict(size=15)))
fig.show()

In [73]:
ncols_y, nrows_y = 3, -(-len(Y_COLS) // 3)
fig = make_subplots(rows=nrows_y, cols=ncols_y, subplot_titles=Y_COLS)
for i, col in enumerate(Y_COLS):
    r, c = divmod(i, ncols_y)
    patho = col.split("_")[-1]
    fig.add_trace(go.Histogram(x=df_sub[col].dropna(), showlegend=False,
                                marker_color=COLORS.get(patho, "#7F8C8D"),
                                xbins=dict(start=0)),  # tous les taux_* sont >= 0
                  row=r + 1, col=c + 1)
fig.update_xaxes(showticklabels=False)
fig.update_annotations(font_size=11)
fig.update_layout(height=260 * nrows_y, width=1400, template="plotly_white",
                   font=dict(size=11),
                   margin=dict(t=90),
                   title=dict(text="Distributions : variables cibles (Y)",
                               font=dict(size=15)))
fig.show()

### Saisonnalité météo

In [71]:
cles_saison = ["temp_moy", "precip_total", "nb_jours_orage", "nb_jours_chaud30"]
df_sais = df_clean.groupby("mois")[cles_saison].mean().reset_index()
df_sais["mois_label"] = df_sais["mois"].apply(lambda m: MOIS_LABELS[m - 1])

fig = make_subplots(rows=1, cols=len(cles_saison), subplot_titles=cles_saison)
for i, col in enumerate(cles_saison, 1):
    fig.add_trace(go.Bar(x=df_sais["mois_label"], y=df_sais[col], marker_color="#2980B9",
                          showlegend=False), row=1, col=i)
fig.update_layout(height=320, width = 1400, template="plotly_white",
                   title="Saisonnalité météo : moyenne nationale par mois")
fig.show()

### Corrélation météo × Y

In [75]:
corr = df_clean[METEO_COLS + Y_COLS].corr().loc[METEO_COLS, Y_COLS]

fig = go.Figure(go.Heatmap(
    z=corr.values, x=corr.columns, y=corr.index,
    colorscale="RdBu", zmid=0, zmin=-1, zmax=1,
    colorbar=dict(title="r"),
    hovertemplate=None
))
fig.update_layout(title="Corrélation météo × Y", template="plotly_white",
                   height=650, xaxis_tickangle=-45)
fig.show()

In [82]:
n_p = len(PATHOLOGIES)
fig = make_subplots(rows=1, cols=n_p, subplot_titles=[f"Top corrélations — {p}" for p in PATHOLOGIES])
for i, p in enumerate(PATHOLOGIES, 1):
    col = f"taux_urgences_{p}"
    top = corr[col].sort_values(key=abs, ascending=True).tail(10)
    bar_colors = ["#E74C3C" if v > 0 else "#2980B9" for v in top.values]
    fig.add_trace(go.Bar(x=top.values, y=top.index, orientation="h",
                          marker_color=bar_colors, showlegend=False), row=1, col=i)
fig.update_yaxes(automargin=True)  # laisse la place aux noms de variables longs sur l'axe Y
fig.update_annotations(font_size=11)
fig.update_layout(height=380, width=1400, template="plotly_white",
                   margin=dict(l=180),
                   title="Top corrélations météo : taux_urgences")
fig.show()

### Météo × urgences — zoom sur la corrélation la plus forte par pathologie

In [79]:
for p in PATHOLOGIES:
    col_y = f"taux_urgences_{p}"
    top_var = corr[col_y].abs().sort_values(ascending=False).index[0]
    d = df_clean[[top_var, "mois", col_y]].dropna().copy()
    d["mois_label"] = d["mois"].apply(lambda m: MOIS_LABELS[m - 1])

    fig = px.scatter(
        d, x=top_var, y=col_y, color="mois_label", trendline="ols",
        title=f"{top_var} × {col_y} (r={corr.loc[top_var, col_y]:.2f})",
        template="plotly_white",
    )
    fig.show()

### Y — saisonnalité par mois

In [85]:
# Heatmap année × mois — les 6 ans sont bien visibles (contrairement à une
# moyenne par mois qui les aurait tous fondus ensemble)
SCALES = {"allergie": "Reds", "asthme": "Blues"}

fig = make_subplots(rows=2, cols=2, subplot_titles=Y_COLS, vertical_spacing=0.15)
positions = [(1, 1), (1, 2), (2, 1), (2, 2)]
for col, (r, c) in zip(Y_COLS, positions):
    patho = col.split("_")[-1]
    piv = (df_clean.groupby(["annee", "mois"])[col].mean()
           .unstack("mois").reindex(columns=range(1, 13)))
    fig.add_trace(go.Heatmap(
        z=piv.values, x=MOIS_LABELS, y=piv.index.astype(str),
        colorscale=SCALES.get(patho, "Greys"),
        hovertemplate="%{y} %{x}<br>%{z:.0f}<extra></extra>",
        showscale=True, colorbar=dict(len=0.42, y=1 if r == 1 else 0,
                                       yanchor="top" if r == 1 else "bottom"),
    ), row=r, col=c)
fig.update_annotations(font_size=11)
fig.update_layout(height=650, width=1000, template="plotly_white",
                   title=dict(text="Y — année × mois (2020-2025)", font=dict(size=15)))
fig.show()

### Y — top 10 des départements

In [86]:
for p in PATHOLOGIES:
    col = f"taux_urgences_{p}"
    df_dept_y = df_clean.groupby("dept")[col].mean().reset_index()
    top10 = df_dept_y.nlargest(10, col).sort_values(col)  # ascendant -> le plus fort en haut du bar horizontal

    fig = go.Figure(go.Bar(
        x=top10[col], y=top10["dept"], orientation="h",
        marker=dict(color=top10[col], colorscale=SCALES.get(p, "Greys"), showscale=True,
                    colorbar=dict(title="taux")),
        hovertemplate="Dept %{y}<br>%{x:.0f}<extra></extra>",
    ))
    fig.update_layout(
        title=f"Top 10 départements — {col} (moyenne 2020-2025)",
        template="plotly_white", height=420, width=700,
        xaxis_title="taux moyen /100k urgences toutes causes",
    )
    fig.show()

### Y — cas où `taux_hosp` > `taux_urgences`

`taux_urgences_X` et `taux_hosp_X` ont des **dénominateurs différents**
(respectivement le total des urgences toutes causes, et le total des
hospitalisations post-urgences toutes causes) — ce ne sont pas des étapes
d'un même entonnoir de patients, donc `taux_hosp_X > taux_urgences_X` n'est
pas logiquement impossible. Mais si ça arrive **systématiquement**, ça veut
dire quelque chose ; si ça arrive **ponctuellement dans de petits
départements**, c'est probablement du bruit d'échantillon (même problème de
petit dénominateur que dans la section 3).

In [89]:
df_clean[df_clean['taux_hosp_asthme'].isna()].groupby(
    ['dept', 'mois']).size()

dept  mois
48    1       1
      2       1
      3       1
55    8       1
      9       1
dtype: int64

In [90]:
for p in PATHOLOGIES:
    u, h = f"taux_urgences_{p}", f"taux_hosp_{p}"
    mask = df_clean[h] > df_clean[u]
    n = int(mask.sum())
    print(f"{p} : {n}/{len(df_clean)} lignes ({n / len(df_clean) * 100:.1f}%) où {h} > {u}")

    sub = df_clean.loc[mask, ["dept", "annee_mois", u, h]].copy()
    sub["pop_totale"] = df.loc[sub.index, "pop_totale"]  # df = df_model brut, chargé en cellule 1
    sub = sub.sort_values(h, ascending=False)

    print(f"  pop_totale moyenne (lignes concernées) : {sub['pop_totale'].mean():,.0f}"
          f" | pop_totale moyenne (ensemble) : {df['pop_totale'].mean():,.0f}")
    display(sub.head(5))
    print()

allergie : 234/6912 lignes (3.4%) où taux_hosp_allergie > taux_urgences_allergie
  pop_totale moyenne (lignes concernées) : 266,500 | pop_totale moyenne (ensemble) : 680,797


,dept,annee_mois,taux_urgences_allergie,taux_hosp_allergie,pop_totale
1519,23,2020-08,2250.73,4638.81,108840
1577,23,2025-06,2073.27,4082.22,108840
3318,48,2020-07,1018.08,3469.02,72720
1567,23,2024-08,2088.39,3312.50,108840
1543,23,2022-08,1965.44,3280.12,108840



asthme : 4795/6912 lignes (69.4%) où taux_hosp_asthme > taux_urgences_asthme
  pop_totale moyenne (lignes concernées) : 771,453 | pop_totale moyenne (ensemble) : 680,797


,dept,annee_mois,taux_urgences_asthme,taux_hosp_asthme,pop_totale
6104,86,2024-09,1926.67,5891.97,422620
2540,37,2021-09,1357.49,5870.11,617070
6716,95,2021-09,2603.69,5749.28,1298030
6740,95,2023-09,2623.14,5549.83,1298030
2541,37,2021-10,1650.79,5398.87,617070


**Lecture** — les deux pathologies donnent des résultats opposés :

- **Allergie : seulement 3.4 % des lignes**, concentrées sur des départements
  peu peuplés (pop. moyenne ~266k vs ~681k pour l'ensemble), et dominées par
  un seul dept (23 — Creuse) qui revient sans cesse dans le top. → **bruit
  de petit échantillon** (même mécanisme que le dénominateur faible identifié
  en section 3), pas un signal à interpréter.
- **Asthme : 69.4 % des lignes**, sans lien avec la taille de la population
  (pop. moyenne ~771k, *au-dessus* de la moyenne générale). Un écart aussi
  large et systématique n'est pas du bruit : il signifie que **l'asthme pèse
  proportionnellement plus lourd parmi les hospitalisations que parmi les
  urgences** — cohérent avec le fait clinique qu'une crise d'asthme, une fois
  aux urgences, a plus de chances de déboucher sur une hospitalisation qu'une
  réaction allergique (généralement traitée et renvoyée). Les cas les plus
  marqués se concentrent en **septembre-novembre**, ce qui recoupe le pic
  bien documenté des exacerbations d'asthme à la rentrée scolaire.

→ Ce test sépare utilement le bruit (allergie/petits depts) du signal
clinique réel (asthme/national) sans avoir besoin d'un seuil arbitraire.

---
## Conclusion

- **Données propres** : les NA viennent d'un manque de couverture de stations
  (pas un bug), et les valeurs "extrêmes" correspondent à des événements réels
  (canicule 2022, froid alpin 01/2021, épisodes cévenols) → rien à corriger,
  juste imputer les NA météo (fait) et laisser les NA Y tels quels.
- **Allergie ↑ avec la chaleur** (`nb_jours_chaud25`, `temp_max/moy` : r≈0.5),
  **asthme ↓ avec la chaleur/le soleil et ↑ avec l'humidité** (r≈0.25-0.32,
  signe inversé) → deux mécanismes climatiques différents, à modéliser
  séparément.
- **Petits départements = ratios Y instables** (dénominateur faible) : à
  garder en tête pour la modélisation (pondération par le volume, ou seuil
  d'exclusion) plutôt qu'à corriger ici.
- **L'asthme est proportionnellement plus hospitalisé que l'allergie**
  (`taux_hosp > taux_urgences` dans 69 % des lignes asthme vs 3 % allergie,
  écart national et non lié à la taille du dept) — cohérent avec une
  sévérité clinique plus élevée, pic net à la rentrée scolaire (sept-nov).
- Corrélations météo individuelles modérées (météo seul n'explique pas tout)
  → à combiner avec pollen / qualité de l'air / contexte pour la
  modélisation.